# Healthcare Operations SQL Validation Notebook

This notebook supports the validation expectations for AI-assisted SQL work. Use it before accepting an AI-generated query as correct. It helps you check schema fit, row counts, date ranges, status values, joins, filters, and expected outputs.

## Validation workflow

Use these cells in order when reviewing AI-generated SQL:

1. Confirm the database connection and available tables.
2. Inspect schemas and row counts.
3. Check fixed reporting dates and valid status values.
4. Check joins before trusting multi-table results.
5. Add filters one at a time.
6. Run expected-output checks or sanity checks.

In [1]:
import sqlite3
import pandas as pd

DB_FILE = "healthcare_operations.db"
conn = sqlite3.connect(DB_FILE)
print(f"Connected to {DB_FILE}")

Connected to healthcare_operations.db


## 1. Table list check

Start by confirming that the database contains the tables your query expects. This helps catch prompts or SQL that assume the wrong database setup.

In [2]:
tables = pd.read_sql("""
SELECT name AS table_name
FROM sqlite_master
WHERE type = 'table'
  AND name NOT LIKE 'sqlite_%'
ORDER BY name;
""", conn)
tables

,table_name
0,appointments
1,clinics
2,patients
3,providers
4,reporting_context


## 2. Schema inspection

Use the schema as the source of truth. Any table or column that does not appear here should be treated as a possible hallucination.

In [3]:
for table_name in tables['table_name']:
    print(f"\n--- {table_name} ---")
    display(pd.read_sql(f"PRAGMA table_info({table_name});", conn))


--- appointments ---


,cid,name,type,notnull,dflt_value,pk
0,0,appointment_id,INTEGER,0,None,1
1,1,patient_id,INTEGER,1,None,0
2,2,provider_id,INTEGER,1,None,0
3,3,clinic_id,INTEGER,1,None,0
4,4,appointment_date,TEXT,1,None,0
5,5,appointment_status,TEXT,1,None,0
6,6,appointment_type,TEXT,1,None,0



--- clinics ---


,cid,name,type,notnull,dflt_value,pk
0,0,clinic_id,INTEGER,0,None,1
1,1,clinic_name,TEXT,1,None,0
2,2,city,TEXT,1,None,0
3,3,region,TEXT,1,None,0



--- patients ---


,cid,name,type,notnull,dflt_value,pk
0,0,patient_id,INTEGER,0,None,1
1,1,patient_name,TEXT,0,None,0
2,2,date_of_birth,TEXT,0,None,0
3,3,city,TEXT,0,None,0
4,4,insurance_type,TEXT,0,None,0



--- providers ---


,cid,name,type,notnull,dflt_value,pk
0,0,provider_id,INTEGER,0,None,1
1,1,provider_name,TEXT,0,None,0
2,2,specialty,TEXT,0,None,0



--- reporting_context ---


,cid,name,type,notnull,dflt_value,pk
0,0,context_id,INTEGER,0,None,1
1,1,reporting_date,TEXT,1,None,0
2,2,data_start_date,TEXT,1,None,0
3,3,data_end_date,TEXT,1,None,0
4,4,recent_30_day_start,TEXT,1,None,0
5,5,recent_60_day_start,TEXT,1,None,0
6,6,recent_3_month_start,TEXT,1,None,0
7,7,recent_6_month_start,TEXT,1,None,0
8,8,primary_date_column,TEXT,1,None,0
9,9,note,TEXT,0,None,0


## 3. Row counts

Check row counts before and after writing complex queries. If a source table has zero rows, a query that returns no rows may be behaving correctly.

In [4]:
row_counts = []
for table_name in tables['table_name']:
    count = pd.read_sql(f"SELECT COUNT(*) AS row_count FROM {table_name};", conn).iloc[0, 0]
    row_counts.append({'table_name': table_name, 'row_count': count})
pd.DataFrame(row_counts)

,table_name,row_count
0,appointments,120
1,clinics,3
2,patients,50
3,providers,10
4,reporting_context,1


## 4. Reporting context and min/max dates

Use the fixed reporting context instead of live-date logic such as `DATE('now')`. This prevents fictional historical datasets from returning empty or misleading results.

In [5]:
pd.read_sql("SELECT * FROM reporting_context;", conn)

,context_id,reporting_date,data_start_date,data_end_date,recent_30_day_start,recent_60_day_start,recent_3_month_start,recent_6_month_start,primary_date_column,note
0,1,2025-06-29,2025-01-06,2025-06-29,2025-05-30,2025-04-30,2025-03-29,2024-12-29,appointments.appointment_date,Use reporting_date instead of date('now') beca...


In [6]:
pd.read_sql("SELECT * FROM dataset_date_range;", conn)

,table_name,date_column,min_date,max_date,row_count
0,appointments,appointment_date,2025-01-06,2025-06-29,120


## 5. Valid status and category values

Check exact spelling and capitalization before writing filters. Small mismatches such as `no-show` vs. `no_show` or `canceled` vs. `Canceled` can change results.

In [7]:
pd.read_sql("SELECT * FROM status_value_counts ORDER BY table_name, status_column, row_count DESC;", conn)

,table_name,status_column,status_value,row_count
0,appointments,appointment_status,completed,87
1,appointments,appointment_status,no_show,23
2,appointments,appointment_status,canceled,10
3,appointments,appointment_type,follow_up,36
4,appointments,appointment_type,consult,32
5,appointments,appointment_type,checkup,29
6,appointments,appointment_type,screening,23


## 6. Join coverage checks

Check whether appointment records match valid patients, providers, and clinics before trusting a multi-table query.

In [8]:
pd.read_sql("""
SELECT
    COUNT(*) AS total_appointments,
    COUNT(p.patient_id) AS appointments_with_matching_patient,
    COUNT(pr.provider_id) AS appointments_with_matching_provider,
    COUNT(c.clinic_id) AS appointments_with_matching_clinic
FROM appointments AS a
LEFT JOIN patients AS p
    ON a.patient_id = p.patient_id
LEFT JOIN providers AS pr
    ON a.provider_id = pr.provider_id
LEFT JOIN clinics AS c
    ON a.clinic_id = c.clinic_id;
""", conn)

,total_appointments,appointments_with_matching_patient,appointments_with_matching_provider,appointments_with_matching_clinic
0,120,120,120,120


## 7. Row-count-before-and-after join checks

These joins should preserve appointment-level rows when each appointment links to one patient, provider, and clinic.

In [9]:
pd.read_sql("""
SELECT 'appointments only' AS query_step, COUNT(*) AS row_count FROM appointments
UNION ALL
SELECT 'appointments + patients', COUNT(*)
FROM appointments AS a
JOIN patients AS p
    ON a.patient_id = p.patient_id
UNION ALL
SELECT 'appointments + patients + providers + clinics', COUNT(*)
FROM appointments AS a
JOIN patients AS p
    ON a.patient_id = p.patient_id
JOIN providers AS pr
    ON a.provider_id = pr.provider_id
JOIN clinics AS c
    ON a.clinic_id = c.clinic_id;
""", conn)

,query_step,row_count
0,appointments only,120
1,appointments + patients,120
2,appointments + patients + providers + clinics,120


## 8. Filter-by-filter debugging

Use exact status values from the database. In this dataset, missed appointments are stored as `no_show`.

In [10]:
pd.read_sql("""
SELECT 'all appointments' AS step, COUNT(*) AS row_count FROM appointments
UNION ALL
SELECT 'last 3 months', COUNT(*)
FROM appointments
WHERE appointment_date >= (SELECT recent_3_month_start FROM reporting_context)
UNION ALL
SELECT 'last 3 months and no_show', COUNT(*)
FROM appointments
WHERE appointment_date >= (SELECT recent_3_month_start FROM reporting_context)
  AND appointment_status = 'no_show';
""", conn)

,step,row_count
0,all appointments,120
1,last 3 months,58
2,last 3 months and no_show,15


## 9. Expected-output check: repeat no-show patients

This validates a common healthcare lab pattern: patients with repeated no-shows in the anchored recent period.

In [11]:
repeat_no_show_patients = pd.read_sql("""
SELECT
    p.patient_id,
    p.patient_name,
    COUNT(a.appointment_id) AS no_show_count
FROM patients AS p
JOIN appointments AS a
    ON p.patient_id = a.patient_id
WHERE a.appointment_date >= (
    SELECT recent_3_month_start
    FROM reporting_context
)
  AND a.appointment_status = 'no_show'
GROUP BY p.patient_id, p.patient_name
HAVING COUNT(a.appointment_id) >= 2
ORDER BY no_show_count DESC, p.patient_name;
""", conn)

expected_columns = ['patient_id', 'patient_name', 'no_show_count']
assert list(repeat_no_show_patients.columns) == expected_columns
repeat_no_show_patients

,patient_id,patient_name,no_show_count
0,28,Brandon Carter,2
1,39,Maya Stewart,2
2,17,Quinn Lewis,2
3,46,Thomas Bell,2


In [12]:
conn.close()
print("Database connection closed.")

Database connection closed.
